# Robot Voice - Compilation Standalone v3

**Version corrigée avec output complet pour debug**

Executez simplement la cellule ci-dessous.

In [ ]:
# ============================================================
# ROBOT VOICE - COMPILATION STANDALONE v3 (Windows DLL)
# Fix: verbose output, meilleure gestion des erreurs
# ============================================================

import os, shutil, glob, subprocess

ROOT = '/content/robot_voice_build'
SRC_DIR = f'{ROOT}/src'
GODOT_CPP = f'{ROOT}/godot-cpp'
FLITE_SRC = f'{ROOT}/flite'
FLITE_OUT = f'{ROOT}/flite_win'
BUILD_DIR = f'{ROOT}/build'
OUT_DIR = f'{ROOT}/output/addons/robot_voice/bin'

# Nettoyer
if os.path.exists(ROOT):
    shutil.rmtree(ROOT)

# === CODE SOURCE C++ (inchangé) ===
ROBOT_VOICE_H = '''#pragma once
#include <godot_cpp/classes/ref_counted.hpp>
#include <godot_cpp/variant/packed_float32_array.hpp>
#include <godot_cpp/variant/string.hpp>

class RobotVoice : public godot::RefCounted {
    GDCLASS(RobotVoice, godot::RefCounted)
private:
    float pitch = 1.45f;
    float volume = 0.85f;
    float chirp_strength = 0.06f;
    float chirp_hz = 1400.0f;
    float chirp_ms = 12.0f;
    int output_rate = 22050;
protected:
    static void _bind_methods();
public:
    RobotVoice();
    ~RobotVoice();
    godot::PackedFloat32Array speak(godot::String text);
    void set_pitch(double p_pitch);
    double get_pitch() const;
    void set_volume(double p_volume);
    double get_volume() const;
    void set_output_rate(int p_rate);
    int get_output_rate() const;
    void set_chirp_strength(double p_value);
    double get_chirp_strength() const;
    void set_chirp_hz(double p_value);
    double get_chirp_hz() const;
    void set_chirp_ms(double p_value);
    double get_chirp_ms() const;
};
'''

ROBOT_VOICE_CPP = '''#include "robot_voice.h"
#include <godot_cpp/core/class_db.hpp>
#include <godot_cpp/core/math.hpp>
#include <godot_cpp/variant/utility_functions.hpp>
#include <flite/flite.h>

using namespace godot;

static bool flite_ready = false;
static cst_voice *flite_voice = nullptr;

static void ensure_flite_voice() {
    if (flite_ready) return;
    flite_init();
    flite_voice = flite_voice_select("slt");
    if (flite_voice == nullptr) flite_voice = flite_voice_select("kal");
    flite_ready = true;
}

RobotVoice::RobotVoice() {}
RobotVoice::~RobotVoice() {}

void RobotVoice::_bind_methods() {
    ClassDB::bind_method(D_METHOD("speak", "text"), &RobotVoice::speak);
    ClassDB::bind_method(D_METHOD("set_pitch", "pitch"), &RobotVoice::set_pitch);
    ClassDB::bind_method(D_METHOD("get_pitch"), &RobotVoice::get_pitch);
    ClassDB::bind_method(D_METHOD("set_volume", "volume"), &RobotVoice::set_volume);
    ClassDB::bind_method(D_METHOD("get_volume"), &RobotVoice::get_volume);
    ClassDB::bind_method(D_METHOD("set_output_rate", "rate"), &RobotVoice::set_output_rate);
    ClassDB::bind_method(D_METHOD("get_output_rate"), &RobotVoice::get_output_rate);
    ClassDB::bind_method(D_METHOD("set_chirp_strength", "value"), &RobotVoice::set_chirp_strength);
    ClassDB::bind_method(D_METHOD("get_chirp_strength"), &RobotVoice::get_chirp_strength);
    ClassDB::bind_method(D_METHOD("set_chirp_hz", "value"), &RobotVoice::set_chirp_hz);
    ClassDB::bind_method(D_METHOD("get_chirp_hz"), &RobotVoice::get_chirp_hz);
    ClassDB::bind_method(D_METHOD("set_chirp_ms", "value"), &RobotVoice::set_chirp_ms);
    ClassDB::bind_method(D_METHOD("get_chirp_ms"), &RobotVoice::get_chirp_ms);
    ADD_PROPERTY(PropertyInfo(Variant::FLOAT, "pitch"), "set_pitch", "get_pitch");
    ADD_PROPERTY(PropertyInfo(Variant::FLOAT, "volume"), "set_volume", "get_volume");
    ADD_PROPERTY(PropertyInfo(Variant::INT, "output_rate"), "set_output_rate", "get_output_rate");
    ADD_PROPERTY(PropertyInfo(Variant::FLOAT, "chirp_strength"), "set_chirp_strength", "get_chirp_strength");
    ADD_PROPERTY(PropertyInfo(Variant::FLOAT, "chirp_hz"), "set_chirp_hz", "get_chirp_hz");
    ADD_PROPERTY(PropertyInfo(Variant::FLOAT, "chirp_ms"), "set_chirp_ms", "get_chirp_ms");
}

PackedFloat32Array RobotVoice::speak(String text) {
    PackedFloat32Array out;
    ensure_flite_voice();
    if (flite_voice == nullptr) {
        UtilityFunctions::push_warning("RobotVoice: flite voice not available.");
        return out;
    }
    CharString utf8 = text.utf8();
    cst_wave *wave = flite_text_to_wave(utf8.get_data(), flite_voice);
    if (wave == nullptr) return out;
    const int in_rate = wave->sample_rate;
    const int in_count = cst_wave_num_samples(wave);
    const int16_t *in_samples = cst_wave_samples(wave);
    if (in_count <= 0 || in_samples == nullptr) { delete_wave(wave); return out; }
    const float safe_pitch = pitch <= 0.01f ? 0.01f : pitch;
    const float rate_scale = static_cast<float>(output_rate) / static_cast<float>(in_rate);
    const int out_count = static_cast<int>((static_cast<float>(in_count) * rate_scale) / safe_pitch);
    if (out_count <= 0) { delete_wave(wave); return out; }
    out.resize(out_count);
    const float inv_scale = static_cast<float>(in_rate) / static_cast<float>(output_rate);
    const float chirp_len_s = chirp_ms * 0.001f;
    const int chirp_len = static_cast<int>(chirp_len_s * output_rate);
    const int chirp_period = static_cast<int>(0.18f * output_rate);
    for (int i = 0; i < out_count; i++) {
        float src = static_cast<float>(i) * safe_pitch * inv_scale;
        int idx = static_cast<int>(src);
        float frac = src - static_cast<float>(idx);
        if (idx >= in_count) { idx = in_count - 1; frac = 0.0f; }
        int idx2 = idx + 1;
        if (idx2 >= in_count) idx2 = in_count - 1;
        float s0 = static_cast<float>(in_samples[idx]) / 32768.0f;
        float s1 = static_cast<float>(in_samples[idx2]) / 32768.0f;
        float sample = (s0 + (s1 - s0) * frac) * volume;
        if (chirp_strength > 0.0f && chirp_period > 0) {
            int pos = i % chirp_period;
            if (pos < chirp_len) {
                float t = static_cast<float>(pos) / static_cast<float>(output_rate);
                sample += Math::sin(2.0f * Math_PI * chirp_hz * t) * chirp_strength;
            }
        }
        sample = Math::clamp(sample, -1.0f, 1.0f);
        out.set(i, sample);
    }
    delete_wave(wave);
    return out;
}

void RobotVoice::set_pitch(double p_pitch) { pitch = static_cast<float>(p_pitch); }
double RobotVoice::get_pitch() const { return pitch; }
void RobotVoice::set_volume(double p_volume) { volume = static_cast<float>(p_volume); }
double RobotVoice::get_volume() const { return volume; }
void RobotVoice::set_output_rate(int p_rate) { output_rate = p_rate; }
int RobotVoice::get_output_rate() const { return output_rate; }
void RobotVoice::set_chirp_strength(double p_value) { chirp_strength = static_cast<float>(p_value); }
double RobotVoice::get_chirp_strength() const { return chirp_strength; }
void RobotVoice::set_chirp_hz(double p_value) { chirp_hz = static_cast<float>(p_value); }
double RobotVoice::get_chirp_hz() const { return chirp_hz; }
void RobotVoice::set_chirp_ms(double p_value) { chirp_ms = static_cast<float>(p_value); }
double RobotVoice::get_chirp_ms() const { return chirp_ms; }
'''

REGISTER_TYPES_CPP = '''#include <godot_cpp/godot.hpp>
#include <godot_cpp/core/class_db.hpp>
#include "robot_voice.h"
using namespace godot;

void initialize_robot_voice(ModuleInitializationLevel p_level) {
    if (p_level != MODULE_INITIALIZATION_LEVEL_SCENE) return;
    ClassDB::register_class<RobotVoice>();
}

void uninitialize_robot_voice(ModuleInitializationLevel p_level) {
    if (p_level != MODULE_INITIALIZATION_LEVEL_SCENE) return;
}

extern "C" GDExtensionBool GDE_EXPORT robot_voice_library_init(
        GDExtensionInterfaceGetProcAddress p_get_proc_address,
        const GDExtensionClassLibraryPtr p_library,
        GDExtensionInitialization *r_initialization) {
    GDExtensionBinding::InitObject init_obj(p_get_proc_address, p_library, r_initialization);
    init_obj.register_initializer(initialize_robot_voice);
    init_obj.register_terminator(uninitialize_robot_voice);
    init_obj.set_minimum_library_initialization_level(MODULE_INITIALIZATION_LEVEL_SCENE);
    return init_obj.init();
}
'''

GDEXTENSION = '''[configuration]
entry_symbol = "robot_voice_library_init"
compatibility_minimum = 4.3
reloadable = true

[libraries]
windows.release.x86_64 = "res://addons/robot_voice/bin/robot_voice.windows.release.x86_64.dll"
windows.debug.x86_64 = "res://addons/robot_voice/bin/robot_voice.windows.release.x86_64.dll"
'''

PLAYER_GD = '''extends Node
class_name RobotVoicePlayer

@export var mix_rate: int = 22050
@export var buffer_seconds: float = 0.5

var _player: AudioStreamPlayer
var _playback: AudioStreamGeneratorPlayback

func _ready() -> void:
    var generator = AudioStreamGenerator.new()
    generator.mix_rate = mix_rate
    generator.buffer_length = buffer_seconds
    _player = AudioStreamPlayer.new()
    _player.stream = generator
    add_child(_player)
    _player.play()
    _playback = _player.get_stream_playback()

func push_mono(samples: PackedFloat32Array) -> void:
    if _playback == null: return
    for s in samples:
        _playback.push_frame(Vector2(s, s))

func push_stereo(samples: PackedVector2Array) -> void:
    if _playback == null: return
    for frame in samples:
        _playback.push_frame(frame)

func reset() -> void:
    if _player == null: return
    _player.stop()
    _player.play()
    _playback = _player.get_stream_playback()
'''

def check_file(path, desc):
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f'  OK: {desc} ({size} bytes)')
        return True
    else:
        print(f'  ERREUR: {desc} MANQUANT!')
        return False

# === ETAPE 1: Creer fichiers source ===
print('=' * 60)
print('ETAPE 1/6: Creation des fichiers source')
print('=' * 60)

os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

with open(f'{SRC_DIR}/robot_voice.h', 'w') as f: f.write(ROBOT_VOICE_H)
with open(f'{SRC_DIR}/robot_voice.cpp', 'w') as f: f.write(ROBOT_VOICE_CPP)
with open(f'{SRC_DIR}/register_types.cpp', 'w') as f: f.write(REGISTER_TYPES_CPP)
print('OK')

# === ETAPE 2: Dependances ===
print('\n' + '=' * 60)
print('ETAPE 2/6: Installation des dependances')
print('=' * 60)

!apt-get update -qq && apt-get install -y -qq build-essential git mingw-w64 cmake ninja-build zip scons
print('OK')

# === ETAPE 3: Clone godot-cpp ===
print('\n' + '=' * 60)
print('ETAPE 3/6: Clone godot-cpp (branch 4.3)')
print('=' * 60)

!git clone --depth=1 --branch 4.3 https://github.com/godotengine/godot-cpp.git {GODOT_CPP}
print('OK')

# === ETAPE 4: Clone et compile flite ===
print('\n' + '=' * 60)
print('ETAPE 4/6: Compile flite pour Windows (~3-5 min)')
print('=' * 60)

!git clone --depth=1 https://github.com/festvox/flite.git {FLITE_SRC}

# Configuration avec variables d'environnement explicites
os.chdir(FLITE_SRC)
os.makedirs(FLITE_OUT, exist_ok=True)

print('\n>>> Configure flite...')
!./configure --host=x86_64-w64-mingw32 --prefix={FLITE_OUT} --disable-shared CC=x86_64-w64-mingw32-gcc CXX=x86_64-w64-mingw32-g++ AR=x86_64-w64-mingw32-ar RANLIB=x86_64-w64-mingw32-ranlib

print('\n>>> Build flite (complet)...')
!make -j$(nproc)

print('\n>>> Install flite...')
!make install

print('\n>>> Verification flite:')
!ls -la {FLITE_OUT}/lib/ 2>/dev/null || echo 'Dossier lib inexistant'
!ls -la {FLITE_OUT}/include/flite/ 2>/dev/null || echo 'Dossier include/flite inexistant'

flite_ok = check_file(f'{FLITE_OUT}/lib/libflite.a', 'libflite.a')
check_file(f'{FLITE_OUT}/lib/libflite_cmu_us_slt.a', 'libflite_cmu_us_slt.a')
check_file(f'{FLITE_OUT}/include/flite/flite.h', 'flite.h')

if not flite_ok:
    print('\n!!! FLITE BUILD FAILED - voir erreurs ci-dessus !!!')
    # Essayer de montrer les erreurs
    print('\nContenu de flite/build:')
    !ls -la {FLITE_SRC}/build/ 2>/dev/null || echo 'Pas de dossier build'

# === ETAPE 5: Compile godot-cpp ===
print('\n' + '=' * 60)
print('ETAPE 5/6: Compile godot-cpp pour Windows (~8-12 min)')
print('=' * 60)

os.chdir(GODOT_CPP)

# Utiliser scons avec tous les flags necessaires pour MinGW
print('\n>>> scons build (output complet)...')
!scons platform=windows target=template_release arch=x86_64 use_mingw=yes verbose=yes -j$(nproc) 2>&1 | head -100

# Si echec, montrer les dernières lignes
print('\n>>> Dernières lignes du build:')
!scons platform=windows target=template_release arch=x86_64 use_mingw=yes -j$(nproc) 2>&1 | tail -30

print('\n>>> Verification godot-cpp:')
!ls -la {GODOT_CPP}/bin/ 2>/dev/null || echo 'Dossier bin inexistant'

godot_lib = f'{GODOT_CPP}/bin/libgodot-cpp.windows.template_release.x86_64.a'
godot_ok = check_file(godot_lib, 'libgodot-cpp.a')

# === ETAPE 6: Compile robot_voice ===
print('\n' + '=' * 60)
print('ETAPE 6/6: Compile robot_voice GDExtension')
print('=' * 60)

if not flite_ok or not godot_ok:
    print('\n!!! IMPOSSIBLE: Dependances manquantes !!!')
    print(f'  Flite: {"OK" if flite_ok else "ECHEC"}')
    print(f'  godot-cpp: {"OK" if godot_ok else "ECHEC"}')
    print('\nVerifiez les erreurs ci-dessus et relancez.')
else:
    # CMakeLists avec chemins explicites
    CMAKELISTS = f'''cmake_minimum_required(VERSION 3.21)
project(robot_voice LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)

add_library(robot_voice SHARED
    {SRC_DIR}/robot_voice.cpp
    {SRC_DIR}/register_types.cpp
)

target_include_directories(robot_voice PRIVATE
    {GODOT_CPP}/include
    {GODOT_CPP}/gen/include
    {GODOT_CPP}/gdextension
    {FLITE_OUT}/include
)

target_link_libraries(robot_voice PRIVATE
    {godot_lib}
    {FLITE_OUT}/lib/libflite_cmu_us_slt.a
    {FLITE_OUT}/lib/libflite_cmu_us_kal.a
    {FLITE_OUT}/lib/libflite_usenglish.a
    {FLITE_OUT}/lib/libflite_cmulex.a
    {FLITE_OUT}/lib/libflite.a
    -lwinmm
    -static-libgcc
    -static-libstdc++
)

target_compile_definitions(robot_voice PRIVATE TYPED_METHOD_BIND)

set_target_properties(robot_voice PROPERTIES
    OUTPUT_NAME "robot_voice.windows.release.x86_64"
    PREFIX ""
    SUFFIX ".dll"
)
'''

    os.makedirs(BUILD_DIR, exist_ok=True)
    with open(f'{BUILD_DIR}/CMakeLists.txt', 'w') as f: f.write(CMAKELISTS)
    
    os.chdir(BUILD_DIR)
    
    print('\n>>> cmake configure...')
    !cmake -G Ninja -DCMAKE_SYSTEM_NAME=Windows -DCMAKE_C_COMPILER=x86_64-w64-mingw32-gcc -DCMAKE_CXX_COMPILER=x86_64-w64-mingw32-g++ .
    
    print('\n>>> ninja build...')
    !ninja -v
    
    # Copier DLL
    dll_found = False
    for dll in glob.glob(f'{BUILD_DIR}/*.dll'):
        shutil.copy(dll, OUT_DIR)
        size = os.path.getsize(dll)
        print(f'\nDLL copiee: {dll} ({size} bytes)')
        dll_found = True
    
    if not dll_found:
        print('\nERREUR: Aucune DLL generee!')
        print('\nContenu du build dir:')
        !ls -la {BUILD_DIR}
    else:
        # Creer fichiers addon
        addon_dir = f'{ROOT}/output/addons/robot_voice'
        with open(f'{addon_dir}/robot_voice.gdextension', 'w') as f: f.write(GDEXTENSION)
        with open(f'{addon_dir}/robot_voice_player.gd', 'w') as f: f.write(PLAYER_GD)
        
        # === PACKAGING ===
        print('\n' + '=' * 60)
        print('PACKAGING')
        print('=' * 60)
        
        OUT_ZIP = f'{ROOT}/robot_voice_addon.zip'
        os.chdir(f'{ROOT}/output')
        !zip -r {OUT_ZIP} addons
        
        print('\nContenu du ZIP:')
        !unzip -l {OUT_ZIP}
        
        # Taille du ZIP
        zip_size = os.path.getsize(OUT_ZIP)
        print(f'\nTaille du ZIP: {zip_size} bytes ({zip_size/1024/1024:.2f} MB)')
        
        if zip_size < 100000:  # Moins de 100KB = probablement pas de DLL
            print('\n!!! ATTENTION: ZIP trop petit, verifiez que la DLL est incluse !!!')
        
        # === TELECHARGEMENT ===
        print('\n' + '=' * 60)
        print('TERMINE! Telechargement...')
        print('=' * 60)
        
        from google.colab import files
        files.download(OUT_ZIP)
        
        print('\n>>> Extrayez robot_voice_addon.zip a la racine de votre projet Godot <<<')